# 시행계획 시도 분리·정제 공통 유틸 사용 템플릿

이 파일을 복사한 뒤 파일명을 `YYYYMMDD_EDA_{연도}_전국_시도분리정제.ipynb` 형식으로 변경한다.

복사 후 반드시 확인할 항목:

1. `YEAR`, `SOURCE_FILE`, `SOURCE_SHEET`
2. 해당 연도의 0원 표기인 `ZERO_BUDGET_TOKENS`
3. 연도 고유 반복 머리글인 `EXTRA_HEADER_PATTERNS` — 2016~2020(제3차 기본계획) 원본은 시도별 단위표기 헤더 행("(단위 : 백만원)" 등)이 있는지 반드시 확인하고, 있으면 `UNIT_NOTATION_PATTERN`을 추가한다 (`src/features/pipeline_common.py` 참고, 이슈 #24)
4. **재원구분 이중계상 여부 `NEEDS_FUNDING_SOURCE_CLEANUP`** — 2016~2019(제3차 기본계획) 원본은 세부사업 하나가 계/국비/지방비(도비·시군비·비예산 등)로 최대 3~4행 중복 표기돼 있다. 원본 `사업분류재정구분` 값이 "계"/"국비"/"지방비" 등인지 확인 후 True로 변경한다 (`drop_exact_duplicate_rows`, `select_total_budget_rows`, 이슈 #26)
5. 소계 QA 허용오차인 `QA_TOLERANCE`
6. LLM 실행 여부와 연도별 체크포인트 경로

> 원본 예산 컬럼은 덮어쓰지 않는다. LLM 셀은 체크포인트와 API 키를 확인하기 전 실행하지 않는다.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.llm_refine import (  # noqa: E402
    call_llm_once,
    clean_sentence,
    needs_llm_rerun,
    numbers_preserved,
)
from src.features.pipeline_common import (  # noqa: E402
    UNIT_NOTATION_PATTERN,
    assign_labels,
    build_subtotal_qa,
    calculate_budget_changes,
    classify_row,
    clean_text,
    drop_exact_duplicate_rows,
    get_sido_dir,
    normalize_budget_type,
    select_total_budget_rows,
    to_numeric_budget,
)
from src.features.text_match import (  # noqa: E402
    check_pattern,
    check_pattern_broad,
    dedup_label,
    extract_via_regex,
    find_near_duplicates,
)

print(f"프로젝트 루트: {PROJECT_ROOT}")

## 1. 연도·입력·예외 설정

아래 셀은 복사한 노트북에서 가장 먼저 수정한다. `비예산`, `·` 등을 0으로 처리할지는 해당 연도 원본을 확인한 뒤 결정한다.

In [ ]:
YEAR = 2020  # 시행계획 연도
PREVIOUS_YEAR = YEAR - 1

CURRENT_BUDGET_COL = f"{YEAR}년 예산"
PREVIOUS_BUDGET_COL = f"{PREVIOUS_YEAR}년 예산"
CURRENT_NUM_COL = f"{YEAR}년_예산_num"
PREVIOUS_NUM_COL = f"{PREVIOUS_YEAR}년_예산_num"

DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
REPORTS_DIR = PROJECT_ROOT / "reports"
CONFIG_DIR = PROJECT_ROOT / "configs"

SOURCE_FILE = (
    DATA_DIR / "칼럼정렬" / "세부사업표추출_2020년도 지방자치단체 저출산고령사회 시행계획 (제3차 기본계획)_칼럼정렬.xlsx"
)
SOURCE_SHEET = "정리본_자동"
TABLE1_FILE = SOURCE_FILE

ZERO_BUDGET_TOKENS = ("-",)
EXTRA_HEADER_PATTERNS = (UNIT_NOTATION_PATTERN,)
NEEDS_FUNDING_SOURCE_CLEANUP = False
QA_TOLERANCE = 0.01
RUN_LLM = False

CHECKPOINT_PATH = INTERIM_DIR / f"{YEAR}_llm_정제_체크포인트.csv"
QA_PATH = REPORTS_DIR / f"{YEAR}_전국_QA_검증결과.csv"
ROW_TYPE_REVIEW_PATH = REPORTS_DIR / f"{YEAR}_행유형_후보_검토.csv"


## 2. 데이터 로드와 기본 확인

In [ ]:
if not SOURCE_FILE.exists():
    raise FileNotFoundError(f"SOURCE_FILE을 실제 파일로 수정하세요: {SOURCE_FILE}")

df_raw = pd.read_excel(SOURCE_FILE, sheet_name=SOURCE_SHEET)

required_columns = {
    "지역",
    "세부사업명",
    "사업분류재정구분",
    CURRENT_BUDGET_COL,
    PREVIOUS_BUDGET_COL,
    "주요내용",
    "원본행",
}
missing_columns = required_columns.difference(df_raw.columns)
if missing_columns:
    raise KeyError(f"입력 파일에 필요한 컬럼이 없습니다: {sorted(missing_columns)}")

expected_shape = (7003, 12)
if df_raw.shape != expected_shape:
    raise ValueError(f"입력 크기가 예상과 다릅니다: {df_raw.shape} != {expected_shape}")

region_count = df_raw["지역"].nunique()
if region_count != 17:
    raise ValueError(f"지역 수가 예상과 다릅니다: {region_count} != 17")

finance_values = set(
    df_raw["사업분류재정구분"].dropna().astype("string").str.strip().unique()
)
if finance_values != {"공통", "자체"}:
    raise ValueError(f"예상하지 못한 사업분류재정구분 값: {sorted(finance_values)}")

print("데이터 크기:", df_raw.shape)
print("지역 수:", region_count)
print("사업분류재정구분:", sorted(finance_values))
display(df_raw.head())

## 3. 행 분류·계층 전파·숫자 변환

소계 QA에 원본 소계 행의 숫자값도 필요하므로 `df_labeled` 전체에 숫자 컬럼을 만든다.

In [ ]:
if NEEDS_FUNDING_SOURCE_CLEANUP:
    df_raw = drop_exact_duplicate_rows(df_raw)
    df_raw = select_total_budget_rows(
        df_raw,
        budget_cols=[CURRENT_BUDGET_COL, PREVIOUS_BUDGET_COL],
        zero_tokens=ZERO_BUDGET_TOKENS,
    )
    print("재원구분(계/국비/지방비) 정리 후 행 수:", len(df_raw))

df_raw["사업행구분"] = df_raw["세부사업명"].apply(
    lambda value: classify_row(value, extra_header_patterns=EXTRA_HEADER_PATTERNS)
)

# 2020년 중분류 제목은 공통 함수가 인식하는 "(공통)/(자체)"가 아니라
# "(공통사업)/(자체사업)"으로 끝나므로 이 노트북에서만 보정한다.
detail_name = df_raw["세부사업명"].astype("string").str.strip()
medium_category_mask = detail_name.str.match(r"^\d+\.", na=False) & detail_name.str.contains(
    r"\((?:공통|자체)사업\)$",
    regex=True,
    na=False,
)

# 경기의 도·시/도비·시비 보조 소계는 상위 중분류 QA에서 중복 집계하지 않도록 제외한다.
auxiliary_subtotal_mask = detail_name.str.match(r"^\d+\.", na=False) & detail_name.str.contains(
    r"\((?:도|시|도비|시비)\s*자체사업\)$",
    regex=True,
    na=False,
)

# 인천·세종·경기의 전체 총계는 세부사업이 아니다.
compact_detail_name = detail_name.str.replace(r"\s+", "", regex=True)
grand_total_mask = compact_detail_name.isin({"총계", "총계(공통사업+자체사업)", "합계"})

# 경남 원본행 7494는 "3. 인구구조 변화..."로 적혀 있으나 뒤따르는 행은 노후사업이고,
# 실제 3번 중분류 소계가 원본행 7651에 다시 나온다. 반복된 오기 머리글로 제외한다.
gyeongnam_mislabelled_heading_mask = df_raw["지역"].eq("경남") & df_raw["원본행"].eq(7494)

df_raw.loc[medium_category_mask, "사업행구분"] = "중분류_소계"
df_raw.loc[
    auxiliary_subtotal_mask | grand_total_mask | gyeongnam_mislabelled_heading_mask,
    "사업행구분",
] = "헤더반복"

row_type_review_mask = (
    detail_name.str.match(UNIT_NOTATION_PATTERN, na=False)
    | auxiliary_subtotal_mask
    | grand_total_mask
    | gyeongnam_mislabelled_heading_mask
)
row_type_review = df_raw.loc[
    row_type_review_mask,
    ["지역", "원본행", "세부사업명", "사업행구분"],
].copy()
row_type_review["판정"] = "제외"
row_type_review["판정근거"] = ""
row_type_review.loc[
    detail_name.str.match(UNIT_NOTATION_PATTERN, na=False), "판정근거"
] = "단위표기 머리글"
row_type_review.loc[auxiliary_subtotal_mask, "판정근거"] = (
    "경기 도·시/도비·시비 보조 소계"
)
row_type_review.loc[grand_total_mask, "판정근거"] = "지역 전체 총계"
row_type_review.loc[gyeongnam_mislabelled_heading_mask, "판정근거"] = (
    "뒤따르는 노후사업 및 원본행 7651의 실제 3번 소계와 중복되는 원본 오기 머리글"
)

if len(row_type_review) != 11 or row_type_review["판정근거"].eq("").any():
    raise ValueError("2020년 행 유형 예외 11건을 다시 확인하세요.")

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
row_type_review.to_csv(ROW_TYPE_REVIEW_PATH, index=False, encoding="utf-8-sig")
print(f"행 유형 검토표 저장 완료: {ROW_TYPE_REVIEW_PATH}")


df_labeled = df_raw.groupby("지역", group_keys=False).apply(assign_labels)
df_labeled["지역"] = df_raw.loc[df_labeled.index, "지역"]
df_labeled[CURRENT_NUM_COL] = to_numeric_budget(
    df_labeled[CURRENT_BUDGET_COL], zero_tokens=ZERO_BUDGET_TOKENS
)
df_labeled[PREVIOUS_NUM_COL] = to_numeric_budget(
    df_labeled[PREVIOUS_BUDGET_COL], zero_tokens=ZERO_BUDGET_TOKENS
)

df_leaf = df_labeled[df_labeled["사업행구분"] == "세부사업"].copy()
print(df_labeled["사업행구분"].value_counts())
print("세부사업 행 수:", len(df_leaf))

## 4. 텍스트 정제와 예산 재계산

In [ ]:
for column in ["세부사업명", "대분류", "중분류"]:
    df_leaf[column] = clean_text(df_leaf[column])

df_leaf["주요내용"] = clean_text(df_leaf["주요내용"], strip_leading_bullet=True)
df_leaf["사업분류재정구분"] = normalize_budget_type(df_leaf["사업분류재정구분"])

budget_changes = calculate_budget_changes(
    df_leaf[CURRENT_NUM_COL],
    df_leaf[PREVIOUS_NUM_COL],
)
df_leaf[["당해예산", "전년도예산", "증감액", "증감율"]] = budget_changes
display(df_leaf[["지역", "세부사업명", "당해예산", "전년도예산", "증감액", "증감율"]].head())

## 5. 중분류 소계 QA와 저장

QA 계산은 공통 함수가 담당하고, 연도별 파일 저장은 노트북에서 담당한다.

In [ ]:
qa = build_subtotal_qa(
    df_labeled,
    budget_col=CURRENT_NUM_COL,
    tolerance=QA_TOLERANCE,
)

qa["원인분류"] = "판정불가"
qa.loc[qa["결과"].eq("일치"), "원인분류"] = "일치"
qa.loc[
    qa["결과"].eq("불일치") & qa["허용기준결과"].eq("허용"),
    "원인분류",
] = "허용범위 내 차이"
qa.loc[qa["허용기준결과"].eq("초과"), "원인분류"] = "원본 소계 불일치"
qa["판정근거"] = ""

ulsan_common_mask = qa["지역"].eq("울산") & qa["중분류"].eq(
    "1. 함께 돌보고 함께 일하는 사회(공통사업)"
)
qa.loc[ulsan_common_mask, "판정근거"] = (
    "원본 소계가 첫 5개 세부사업 합계 83,577백만원을 누락"
)

ulsan_own_mask = qa["지역"].eq("울산") & qa["중분류"].eq(
    "3. 인구구조 변화 적극 대비 (자체사업)"
)
qa.loc[ulsan_own_mask, "판정근거"] = (
    "원본 소계가 다둥이 행복렌트카 지원 50백만원을 누락"
)

gyeongnam_own_mask = qa["지역"].eq("경남") & qa["중분류"].eq(
    "2. 함께 만들어가는 행복한 노후(자체사업)"
)
qa.loc[gyeongnam_own_mask, "판정근거"] = (
    "원본행 7494의 잘못된 3번 머리글과 뒤 노후사업 구간으로 원본 소계가 29,993백만원 과대"
)

qa_over = qa[qa["허용기준결과"].eq("초과")]
if len(qa) != 100 or len(qa_over) != 3 or qa_over["판정근거"].eq("").any():
    raise ValueError("2020년 QA 기준 수치가 달라졌습니다.")

display(qa["원인분류"].value_counts(dropna=False))
display(qa_over.sort_values("오차율(%)", key=lambda series: series.abs(), ascending=False))

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
qa.to_csv(QA_PATH, index=False, encoding="utf-8-sig")
print(f"QA 저장 완료: {QA_PATH}")

## 6. 주요내용 라벨 정리·부분 추출·유사 사업명 후보

In [ ]:
df_leaf["주요내용"] = df_leaf["주요내용"].apply(dedup_label)
df_leaf["주요내용_패턴"] = df_leaf["주요내용"].apply(check_pattern)
df_leaf["주요내용_패턴_확장"] = df_leaf["주요내용"].apply(check_pattern_broad)

regex_extracted = pd.DataFrame(
    df_leaf["주요내용"].apply(extract_via_regex).tolist(),
    index=df_leaf.index,
)
df_leaf["지원대상"] = regex_extracted["지원대상"]
df_leaf["지원내용_상세"] = regex_extracted["지원내용"]

near_duplicates = find_near_duplicates(df_leaf, threshold=90)
print("유사 사업명 검토 후보:", len(near_duplicates))
display(near_duplicates.head(20))

## 7. 원본 Table 1 대조(필요 시)

원본 파일의 컬럼 배치가 다르면 `column_indices`를 해당 연도에 맞게 변경한다.

In [ ]:
# 예시: 실제 행 번호로 교체한 뒤 주석을 해제한다.
# df_table1 = pd.read_excel(TABLE1_FILE, sheet_name="Table 1", header=None)
# display(show_table1_around(df_table1, center_excel_row=100, window=3, label="원본 대조"))

## 8. LLM 보존형 교정(선택)

`RUN_LLM=True`로 바꾸기 전에 `.env`, 설정 파일, 연도별 체크포인트 파일명을 확인한다. 체크포인트는 인덱스뿐 아니라 지역·원본행·세부사업명까지 대조한 뒤 재사용한다.

In [ ]:
df_leaf_final = df_leaf.copy()
df_leaf_final["연도"] = YEAR
df_leaf_final["주요내용_정제"] = df_leaf_final["주요내용"]

if RUN_LLM:
    import os
    from functools import partial

    import yaml
    from dotenv import load_dotenv
    from openai import OpenAI

    load_dotenv(PROJECT_ROOT / ".env")
    api_key = os.getenv("UPSTAGE_API_KEY")
    if not api_key:
        raise RuntimeError("UPSTAGE_API_KEY가 없습니다.")

    with (CONFIG_DIR / "llm_extraction.yaml").open(encoding="utf-8") as file:
        llm_cfg = yaml.safe_load(file)

    client = OpenAI(api_key=api_key, base_url=llm_cfg["upstage"]["base_url"])
    call_once = partial(call_llm_once, client=client, llm_config=llm_cfg)

    identity_cols = ["지역", "원본행", "세부사업명"]
    checkpoint_cols = [*identity_cols, "주요내용_정제"]

    if CHECKPOINT_PATH.exists():
        checkpoint_df = pd.read_csv(CHECKPOINT_PATH, encoding="utf-8-sig", index_col=0)
        missing_checkpoint_cols = set(checkpoint_cols).difference(checkpoint_df.columns)
        if missing_checkpoint_cols:
            raise KeyError(
                f"체크포인트 신원 검증 컬럼이 없습니다: {sorted(missing_checkpoint_cols)}"
            )
        checkpoint_df.index = pd.to_numeric(checkpoint_df.index, errors="raise").astype(int)
        checkpoint_df = checkpoint_df.loc[~checkpoint_df.index.duplicated(keep="last")]
        checkpoint_df = checkpoint_df.loc[
            checkpoint_df.index.intersection(df_leaf_final.index)
        ]

        common_index = checkpoint_df.index.intersection(df_leaf_final.index)
        current_identity = (
            df_leaf_final.loc[common_index, identity_cols].astype("string").fillna("<NA>")
        )
        checkpoint_identity = (
            checkpoint_df.loc[common_index, identity_cols].astype("string").fillna("<NA>")
        )
        identity_matches = current_identity.eq(checkpoint_identity).all(axis=1)
        invalid_index = identity_matches.index[~identity_matches]
        rerun_index = checkpoint_df.index[
            checkpoint_df["주요내용_정제"].apply(needs_llm_rerun)
        ]
        checkpoint_df = checkpoint_df.drop(
            index=invalid_index.union(rerun_index),
            errors="ignore",
        )
        print(
            "체크포인트 재사용:",
            len(checkpoint_df),
            "건 / 신원 불일치·재실행:",
            len(invalid_index.union(rerun_index)),
            "건",
        )
    else:
        checkpoint_df = pd.DataFrame(columns=checkpoint_cols)

    targets = df_leaf_final.index.difference(checkpoint_df.index, sort=False)
    pending_rows = []

    for position, index in enumerate(targets, start=1):
        row = df_leaf_final.loc[index]
        pending_rows.append(
            {
                "_index": index,
                "지역": row["지역"],
                "원본행": row["원본행"],
                "세부사업명": row["세부사업명"],
                "주요내용_정제": clean_sentence(
                    row["세부사업명"],
                    row["주요내용"],
                    call_once=call_once,
                ),
            }
        )

        if position % 200 == 0:
            partial = pd.DataFrame(pending_rows).set_index("_index")
            checkpoint_df = pd.concat([checkpoint_df, partial])
            checkpoint_df = checkpoint_df.loc[
                ~checkpoint_df.index.duplicated(keep="last")
            ].sort_index()
            checkpoint_df.to_csv(CHECKPOINT_PATH, encoding="utf-8-sig")
            pending_rows = []

    if pending_rows:
        partial = pd.DataFrame(pending_rows).set_index("_index")
        checkpoint_df = pd.concat([checkpoint_df, partial])
        checkpoint_df = checkpoint_df.loc[
            ~checkpoint_df.index.duplicated(keep="last")
        ].sort_index()
        checkpoint_df.to_csv(CHECKPOINT_PATH, encoding="utf-8-sig")

    missing_checkpoint_index = df_leaf_final.index.difference(
        checkpoint_df.index, sort=False
    )
    if len(missing_checkpoint_index):
        raise ValueError(f"체크포인트 누락 인덱스: {missing_checkpoint_index.tolist()}")

    df_leaf_final["주요내용_정제"] = checkpoint_df.loc[
        df_leaf_final.index, "주요내용_정제"
    ]
    print("LLM 신규 처리:", len(targets), "건")
else:
    print("RUN_LLM=False: LLM 호출 없이 주요내용 원문을 유지합니다.")

df_leaf_final["숫자보존"] = df_leaf_final.apply(
    lambda row: numbers_preserved(row["주요내용"], row["주요내용_정제"]),
    axis=1,
)
bad_number_index = df_leaf_final.index[~df_leaf_final["숫자보존"]]

if len(bad_number_index):
    number_review = df_leaf_final.loc[
        bad_number_index,
        ["지역", "원본행", "세부사업명", "주요내용", "주요내용_정제"],
    ].copy()
    number_review["판정"] = "원문 유지"
    number_review["판정근거"] = "LLM 숫자 시퀀스 불일치"
    number_review.to_csv(
        REPORTS_DIR / f"{YEAR}_LLM_숫자불일치_검토.csv",
        encoding="utf-8-sig",
    )
    df_leaf_final.loc[bad_number_index, "주요내용_정제"] = df_leaf_final.loc[
        bad_number_index, "주요내용"
    ]

    if RUN_LLM:
        checkpoint_df.loc[bad_number_index, "주요내용_정제"] = df_leaf_final.loc[
            bad_number_index, "주요내용"
        ]
        checkpoint_df.to_csv(CHECKPOINT_PATH, encoding="utf-8-sig")

    df_leaf_final["숫자보존"] = df_leaf_final.apply(
        lambda row: numbers_preserved(row["주요내용"], row["주요내용_정제"]),
        axis=1,
    )

if not df_leaf_final["숫자보존"].all():
    raise ValueError("숫자 보존 복구 후에도 불일치가 남았습니다.")

print("숫자보존 검증:", int(df_leaf_final["숫자보존"].sum()), "건 통과")

## 9. 시도별 최종 저장

wide 15개 컬럼, long 15개 컬럼, 필터링 전 전체 원본을 각각 한 번만 저장한다.

In [ ]:
output_cols = [
    "연도",
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "세부사업명",
    "주요내용",
    "주요내용_정제",
    "당해예산",
    "전년도예산",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
]
missing_output_cols = set(output_cols).difference(df_leaf_final.columns)
if missing_output_cols:
    raise KeyError(f"최종 저장 전에 필요한 컬럼을 확인하세요: {sorted(missing_output_cols)}")

wide_paths = []
long_paths = []
raw_paths = []

for sido, group in df_leaf_final.groupby("지역"):
    sido_dir = get_sido_dir(INTERIM_DIR, sido)
    output_path = sido_dir / f"{YEAR}_{sido}_세부사업_정제.csv"
    group[output_cols].to_csv(output_path, index=False, encoding="utf-8-sig")
    wide_paths.append(output_path)

    raw_path = sido_dir / f"{YEAR}_{sido}_필터링전_전체원본.csv"
    df_labeled.loc[df_labeled["지역"].eq(sido)].to_csv(
        raw_path, index=False, encoding="utf-8-sig"
    )
    raw_paths.append(raw_path)
    print(f"{sido}: {len(group)}행 저장 -> {output_path}")

id_cols = [
    "연도",
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "세부사업명",
    "주요내용",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
    "주요내용_정제",
]
df_long = df_leaf_final.melt(
    id_vars=id_cols,
    value_vars=["당해예산", "전년도예산"],
    var_name="예산구분",
    value_name="예산액",
)
previous_budget_mask = df_long["예산구분"].eq("전년도예산")
df_long.loc[previous_budget_mask, "연도"] -= 1
df_long.loc[previous_budget_mask, ["증감액", "증감율"]] = None
df_long = df_long.sort_values(["지역", "원본행", "예산구분"]).reset_index(drop=True)

for sido, group in df_long.groupby("지역"):
    sido_dir = get_sido_dir(INTERIM_DIR, sido)
    long_path = sido_dir / f"{YEAR}_{sido}_세부사업_정제_long.csv"
    group.to_csv(long_path, index=False, encoding="utf-8-sig")
    long_paths.append(long_path)

if len(df_leaf_final) != 6858 or df_leaf_final["지역"].nunique() != 17:
    raise ValueError("wide 행 수 또는 지역 수가 2020년 기준과 다릅니다.")
if list(df_leaf_final[output_cols].columns) != output_cols or len(output_cols) != 15:
    raise ValueError("wide 공통 15개 컬럼을 확인하세요.")
if len(df_long) != 13716 or len(df_long.columns) != 15:
    raise ValueError("long 행 수 또는 15개 컬럼을 확인하세요.")
if not (len(wide_paths) == len(long_paths) == len(raw_paths) == 17):
    raise ValueError("시도별 산출물 파일 수가 17개가 아닙니다.")

all_output_paths = [*wide_paths, *long_paths, *raw_paths, QA_PATH, ROW_TYPE_REVIEW_PATH]
non_bom_paths = [
    path for path in all_output_paths if not path.read_bytes().startswith(b"\xef\xbb\xbf")
]
if non_bom_paths:
    raise ValueError(f"UTF-8-SIG가 아닌 산출물: {non_bom_paths}")

print("wide:", len(wide_paths), "개 /", len(df_leaf_final), "행")
print("long:", len(long_paths), "개 /", len(df_long), "행")
print("필터링 전 원본:", len(raw_paths), "개 /", len(df_labeled), "행")

## 완료 전 체크리스트

- [ ] 17개 시도 포함 여부 확인
- [ ] 행 유형 분포 확인
- [ ] 연도 고유 헤더·연속행 후보 원본 대조
- [ ] 재원구분(계/국비/지방비) 이중계상 여부 확인 및 `NEEDS_FUNDING_SOURCE_CLEANUP` 반영 (2016~2019)
- [ ] 중분류 소계 QA 불일치 원인 기록
- [ ] 증감액·증감율 재계산 결과 확인
- [ ] LLM 숫자 보존 불일치 원본 대조
- [ ] wide/long 행 수와 컬럼 스키마 확인
- [ ] 최종 CSV `utf-8-sig` 저장 확인
- [ ] 커널 재시작 후 전체 실행
- [ ] 리팩터링 전후 QA 수치 비교